In [4]:
#CUDA-Based Parallel Vector Addition Using Numba in Python
from numba import cuda
import numpy as np

@cuda.jit
def add_arrays(a, b, c):
    i = cuda.grid(1)             
    if i < c.size:              
        c[i] = a[i] + b[i]

n = 1024

a = np.arange(n, dtype=np.float32)
b = np.arange(n, dtype=np.float32)

c = np.zeros(n, dtype=np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)

threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

add_arrays[blocks_per_grid, threads_per_block](d_a, d_b, d_c)

cuda.synchronize()

c = d_c.copy_to_host()

# Print result
print("First 10 results:", c[:10])

First 10 results: [ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [5]:
#2D Parallel Matrix Multiplication Using CUDA (Numba) in Python
from numba import cuda
import numpy as np
# CUDA Kernel
@cuda.jit
def matmul_2d(A, B, C):
    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:
        temp = 0.0
        for k in range(A.shape[1]):
            temp += A[row, k] * B[k, col]
        C[row, col] = temp

# Host Code
M, K, N = 4, 3, 5   # A: MxK, B: KxN, C: MxN

A = np.arange(M * K, dtype=np.float32).reshape(M, K)
B = np.arange(K * N, dtype=np.float32).reshape(K, N)
C = np.zeros((M, N), dtype=np.float32)

# Copy to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

# 2D CUDA configuration
threads_per_block = (16, 16)
blocks_per_grid_x = (N + threads_per_block[1] - 1) // threads_per_block[1]
blocks_per_grid_y = (M + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid = (blocks_per_grid_y, blocks_per_grid_x)

# Launch kernel
matmul_2d[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
cuda.synchronize()


# Copy result back
C = d_C.copy_to_host()

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C = A x B:\n", C)

Matrix A:
 [[ 0.  1.  2.]
 [ 3.  4.  5.]
 [ 6.  7.  8.]
 [ 9. 10. 11.]]
Matrix B:
 [[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]]
Matrix C = A x B:
 [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
